In [2]:
# Cell 1: Environment Setup & Surrogate Models
import os
import numpy as np
import pandas as pd
from typing import Dict, Any, Tuple
from scipy.optimize import minimize

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor

DATASET_PATH = 'data/SYNTHETIC_DATASET_100_CLEANED.csv'
df = pd.read_csv(DATASET_PATH)

ignore_cols = ['id', 'Sound', 'Mtrue', 'TEtrue', 'Mhat_v01', 'TEhat_v01', 'Soundhat_v01', 'Mat']
feature_cols = [c for c in df.columns if c not in ignore_cols]
X = df[feature_cols]

# Trained Models
model_M = GradientBoostingRegressor(n_estimators=100, max_depth=3, learning_rate=0.08, random_state=42)
model_M.fit(X, df['Mtrue'])

model_TE = Pipeline([('scaler', StandardScaler()), ('reg', Ridge(alpha=5.0))])
model_TE.fit(X, df['TEtrue'])

model_Sound = Pipeline([('scaler', StandardScaler()), ('reg', Ridge(alpha=5.0))])
model_Sound.fit(X, df['Sound'])

def calculate_deterministic_gear_physics(inputs: Dict[str, Any]) -> Dict[str, Any]:
    mn = float(inputs['mn'])
    z1 = int(inputs['z1'])
    z2 = int(inputs['z2'])
    beta_deg = float(inputs['beta'])
    alpha_deg = float(inputs.get('alpha', 20.0))
    b = float(inputs['b'])
    T1 = float(inputs['T1'])
    n1 = float(inputs['n1'])
    
    beta_rad = np.radians(beta_deg)
    alpha_n_rad = np.radians(alpha_deg)
    alpha_t_rad = np.arctan(np.tan(alpha_n_rad) / np.cos(beta_rad))
    
    d1 = (mn * z1) / np.cos(beta_rad)
    d2 = (mn * z2) / np.cos(beta_rad)
    a = (d1 + d2) / 2.0
    v = (np.pi * d1 * n1) / 60000.0
    Ft = (2000.0 * T1) / d1
    cos_beta_b = np.cos(beta_rad) * np.cos(alpha_t_rad) / np.cos(alpha_n_rad)
    wt = Ft / (b * cos_beta_b)
    fm = (n1 * z1) / 60.0
    
    eps_beta = (b * np.sin(beta_rad)) / (np.pi * mn)
    eps_alpha = 1.88 - 3.2 * (1.0 / z1 + 1.0 / z2)
    eps_gamma = eps_alpha + eps_beta
    
    P = float(inputs.get('P', 0.0))
    if P <= 0.0:
        P = (T1 * n1) / 9550.0

    results = dict(inputs)
    results.update({
        'd1_mm': float(round(d1, 4)),
        'd2_mm': float(round(d2, 4)),
        'a': float(round(a, 3)),
        'v_mps': float(round(v, 3)),
        'Ft_N': float(round(Ft, 3)),
        'wt_N_per_mm': float(round(wt, 3)),
        'fm_Hz': float(round(fm, 2)),
        'eps_alpha': float(round(max(1.1, eps_alpha), 3)),
        'eps_beta': float(round(max(0.5, eps_beta), 3)),
        'eps_gamma': float(round(max(1.6, eps_gamma), 3)),
        'P': float(round(P, 2))
    })
    return results

def evaluate_forward_full(inputs: Dict[str, Any]) -> Dict[str, Any]:
    full_inputs = calculate_deterministic_gear_physics(inputs)
    sample = df[feature_cols].iloc[0].copy()
    for k, v in full_inputs.items():
        if k in sample:
            sample[k] = v
    X_eval = pd.DataFrame([sample])
    m = float(model_M.predict(X_eval)[0])
    te = float(model_TE.predict(X_eval)[0])
    s = float(model_Sound.predict(X_eval)[0])
    
    m_pass = (20.0 <= m <= 100.0)
    te_pass = (0.5 <= te <= 2.5)
    s_pass = (50.0 <= s <= 70.0)
    all_pass = (m_pass and te_pass and s_pass)
    
    return {
        'Misalignment_um': round(m, 2),
        'Transmission_Error_um': round(te, 3),
        'Sound_Power_dB': round(s, 2),
        'All_Pass': all_pass
    }

print("Stage 4 forward evaluation engine initialized.")


Stage 4 forward evaluation engine initialized.


In [3]:
# Cell 2: Inverse Sizing Engine
def solve_inverse_gear_parameters(
    operating_conditions: Dict[str, float],
    target_requirements: Dict[str, float],
    initial_guess: Dict[str, float] = None
) -> Dict[str, Any]:
    """
    Multi-Objective Inverse Optimizer:
    Calculates the optimal macro-geometry (mn, z1, beta, b) and micro-geometry (Calpha, Cbeta)
    that satisfies the required Misalignment, TE, and Sound Power targets simultaneously.
    """
    T1 = operating_conditions['T1']
    n1 = operating_conditions['n1']
    gear_ratio = operating_conditions['i']
    
    target_M = target_requirements.get('target_M_um', 45.0)
    target_TE = target_requirements.get('target_TE_um', 1.00)
    target_Sound = target_requirements.get('target_Sound_dB', 55.0)
    
    # Bounded parameter space: [mn, z1, beta, b, Calpha, Cbeta]
    bounds = [
        (1.5, 3.0),    # Normal module mn [mm]
        (16.0, 26.0),  # Pinion teeth z1
        (18.0, 32.0),  # Helix angle beta [deg]
        (22.0, 50.0),  # Face width b [mm]
        (4.0, 20.0),   # Tip relief Calpha [um]
        (4.0, 16.0)    # Crowning Cbeta [um]
    ]
    
    x0 = [2.0, 18.0, 22.0, 32.0, 8.0, 8.0]
    
    def objective_loss(vars_vector):
        mn_val, z1_val, beta_val, b_val, ca_val, cb_val = vars_vector
        z1_int = int(round(z1_val))
        z2_int = int(round(z1_int * gear_ratio))
        
        test_deck = {
            'mn': mn_val,
            'z1': z1_int,
            'z2': z2_int,
            'beta': beta_val,
            'alpha': 20.0,
            'b': b_val,
            'T1': T1,
            'n1': n1,
            'i': gear_ratio,
            'Calpha': ca_val,
            'Cbeta': cb_val,
            'Q': 5,
            'K_A': 1.13
        }
        res = evaluate_forward_full(test_deck)
        m = res['Misalignment_um']
        te = res['Transmission_Error_um']
        s = res['Sound_Power_dB']
        
        penalty = 0.0
        # Enforce specification bounds and targets
        if m < 20.0: penalty += (20.0 - m)**2 * 10.0
        if m > target_M: penalty += (m - target_M)**2 * 10.0
        
        if te < 0.5: penalty += (0.5 - te)**2 * 500.0
        if te > target_TE: penalty += (te - target_TE)**2 * 500.0
        
        if s < 50.0: penalty += (50.0 - s)**2 * 50.0
        if s > target_Sound: penalty += (s - target_Sound)**2 * 50.0
        
        return penalty

    opt_result = minimize(objective_loss, x0, method='L-BFGS-B', bounds=bounds)
    opt_mn, opt_z1, opt_beta, opt_b, opt_ca, opt_cb = opt_result.x
    final_z1 = int(round(opt_z1))
    final_z2 = int(round(final_z1 * gear_ratio))
    
    verified_deck = {
        'mn': round(opt_mn, 2),
        'z1': final_z1,
        'z2': final_z2,
        'beta': round(opt_beta, 1),
        'alpha': 20.0,
        'b': round(opt_b, 1),
        'T1': T1,
        'n1': n1,
        'i': gear_ratio,
        'Calpha': round(opt_ca, 1),
        'Cbeta': round(opt_cb, 1),
        'Q': 5,
        'K_A': 1.13
    }
    verified_outputs = evaluate_forward_full(verified_deck)
    
    return {
        'Optimization_Converged': opt_result.success,
        'Recommended_Parameters': verified_deck,
        'Target_Requirements': target_requirements,
        'Verified_NVH_Outputs': verified_outputs
    }

print("Robust multi-objective inverse optimization solver compiled.")


Robust multi-objective inverse optimization solver compiled.


In [4]:
# Cell 3: Solve Inverse Problem on Target NVH Specifications
op_conditions = {
    'T1': 43.5,
    'n1': 12200.0,
    'i': 6.56
}

design_targets = {
    'target_M_um': 40.0,
    'target_TE_um': 0.95,
    'target_Sound_dB': 53.5
}

print("Running Inverse Optimization Solver...")
solution = solve_inverse_gear_parameters(op_conditions, design_targets)

print("\n================ INVERSE SIZING ENGINE RESULTS ================")
print(f"Optimization Converged: {solution['Optimization_Converged']}")

print("\n[RECOMMENDED GEAR SIZING PARAMETERS]")
for p_name, p_val in solution['Recommended_Parameters'].items():
    print(f"  • {p_name:25s}: {p_val}")

print("\n[VERIFIED FORWARD NVH PERFORMANCE]")
perf = solution['Verified_NVH_Outputs']
print(f"  • Predicted Misalignment:       {perf['Misalignment_um']} µm  (Target <= {design_targets['target_M_um']} µm)")
print(f"  • Predicted Transmission Error: {perf['Transmission_Error_um']} µm (Target <= {design_targets['target_TE_um']} µm)")
print(f"  • Predicted Sound Power Level:  {perf['Sound_Power_dB']} dB  (Target <= {design_targets['target_Sound_dB']} dB)")
print(f"  • Gate Status:                  {'PASS - ALL GATES SATISFIED' if perf['All_Pass'] else 'FAIL'}")
print("===============================================================")

assert perf['All_Pass'] is True, "Output gates violated!"
print("\nVERDICT: STAGE 4 INVERSE ENGINE COMPLETE. ALL GATES CLEARED!")


Running Inverse Optimization Solver...

================ INVERSE SIZING ENGINE RESULTS ================
Optimization Converged: True

[RECOMMENDED GEAR SIZING PARAMETERS]
  • mn                       : 2.0
  • z1                       : 18
  • z2                       : 118
  • beta                     : 22.0
  • alpha                    : 20.0
  • b                        : 32.0
  • T1                       : 43.5
  • n1                       : 12200.0
  • i                        : 6.56
  • Calpha                   : 8.0
  • Cbeta                    : 8.0
  • Q                        : 5
  • K_A                      : 1.13

[VERIFIED FORWARD NVH PERFORMANCE]
  • Predicted Misalignment:       31.34 µm  (Target <= 40.0 µm)
  • Predicted Transmission Error: 0.875 µm (Target <= 0.95 µm)
  • Predicted Sound Power Level:  52.7 dB  (Target <= 53.5 dB)
  • Gate Status:                  PASS - ALL GATES SATISFIED

VERDICT: STAGE 4 INVERSE ENGINE COMPLETE. ALL GATES CLEARED!
